In [ ]:
import pandas as pd
import numpy as np

train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")

In [ ]:
selected_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

## Correlation Filtering

In [ ]:
corr = train_woe.drop(
    columns=["SeriousDlqin2yrs"]
).corr()

corr

In [ ]:
#check for correlation between variables above 0.7
corr_pairs = (
    corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = [
    "variable_1",
    "variable_2",
    "correlation"
]

corr_pairs[
    abs(corr_pairs["correlation"]) > 0.7
].sort_values(
    by="correlation",
    ascending=False
)

## VIF Analysis

In [ ]:
!pip install statsmodels

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
X = train_woe.drop(
    columns=["SeriousDlqin2yrs"]
)

In [ ]:
vif_table = pd.DataFrame()

vif_table["variable"] = X.columns

vif_table["VIF"] = [
    variance_inflation_factor(
        X.values,
        i
    )
    for i in range(X.shape[1])
]

vif_table.sort_values(
    by="VIF",
    ascending=False
)

No material multicollinearity identified.

## Logistic Significance

In [ ]:
import statsmodels.api as sm

In [ ]:
X = train_woe.drop(
    columns=["SeriousDlqin2yrs"]
)

y = train_woe["SeriousDlqin2yrs"]

In [ ]:
X = sm.add_constant(X)

In [ ]:
logit_model = sm.Logit(
    y,
    X
)

result = logit_model.fit()

In [ ]:
result.summary()

The final model contains eight predictors.

All variables are statistically significant at the 5% level.

No material multicollinearity was identified, with all VIF values below 2.

Although NumberOfOpenCreditLinesAndLoans exhibits a positive coefficient after WOE transformation, the variable remains statistically significant and does not show evidence of multicollinearity. Therefore it was retained in the final model.

In [ ]:
final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]